# Module 1b - Create the Lakeflow Declarative Pipeline (optional)

This is an **optional** alternate ingestion path that builds the same bronze/silver/gold tables as `01_ingest_advisories` but as a managed Lakeflow Declarative Pipeline (formerly Delta Live Tables). The point of running it: once the pipeline exists, the instructor can open it from the Pipelines sidebar and walk through the **DLT graph editor** with the audience.

Source code: [`01b_ingest_dlt.py`](./01b_ingest_dlt.py).

What this notebook does:
1. Looks up an existing per-user pipeline by name (idempotent re-runs).
2. Creates a serverless DLT pipeline pointing at `01b_ingest_dlt.py` if not found.
3. Triggers an update.

Skip this notebook if you don't want to demo DLT. The compound agent in module 5 reads from the tables produced by `01_ingest_advisories`, not from the DLT-built tables, so nothing else depends on this.

In [ ]:
%run ./_config


In [ ]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.pipelines import PipelineLibrary, NotebookLibrary
import time, os

w = WorkspaceClient()
PIPELINE_NAME = f"disa-cti-ingest-{USER_SUFFIX}"[:60]

ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
DLT_SOURCE = os.path.dirname(ctx) + "/01b_ingest_dlt"
print(f"DLT source: {DLT_SOURCE}")
print(f"Pipeline:   {PIPELINE_NAME}")


In [ ]:
existing = next((p for p in w.pipelines.list_pipelines() if p.name == PIPELINE_NAME), None)

if existing:
    pipeline_id = existing.pipeline_id
    print(f"Reusing existing pipeline {pipeline_id}")
else:
    created = w.pipelines.create(
        name=PIPELINE_NAME,
        catalog=CATALOG,
        target=SCHEMA,
        serverless=True,
        continuous=False,
        channel="CURRENT",
        photon=True,
        libraries=[PipelineLibrary(notebook=NotebookLibrary(path=DLT_SOURCE))],
        configuration={
            "workshop.catalog": CATALOG,
            "workshop.schema": SCHEMA,
        },
    )
    pipeline_id = created.pipeline_id
    print(f"Created pipeline {pipeline_id}")

cfg_set("dlt_pipeline_id", pipeline_id)
print(f"\nOpen in UI: {w.config.host.rstrip('/')}/pipelines/{pipeline_id}")


In [ ]:
# Trigger one update; print the run URL.
update = w.pipelines.start_update(pipeline_id=pipeline_id, full_refresh=False)
print(f"Update started: {update.update_id}")
print(f"Watch in UI:    {w.config.host.rstrip('/')}/pipelines/{pipeline_id}/updates/{update.update_id}")

## After the pipeline succeeds

Open the pipeline in the workspace (link printed above). The **DLT editor** view shows:

- A graph: `bronze_advisory_files` -> `silver_parsed_advisories` -> `gold_advisories` + `gold_advisory_kev_matches`.
- The expectation `has_parsed` (drops rows where `ai_parse_document` returned null).
- Per-table metrics (rows in/out, expectations, latency).
- The source code panel where you can edit `01b_ingest_dlt.py` directly.

Tables produced live alongside the regular ingestion outputs:

| Path | Source |
|---|---|
| `<schema>.parsed_advisories` | notebook `01_ingest_advisories` (one-shot SQL) |
| `<schema>.silver_parsed_advisories` | this DLT pipeline |
| `<schema>.advisories` | notebook `01_ingest_advisories` (view) |
| `<schema>.gold_advisories` | this DLT pipeline (table) |

Either set is queryable. The compound agent's Genie tool reads from `kev_catalog` / `cves` (which both notebooks share), not from the advisory tables, so neither path is required for the rest of the workshop.